# خُلاصة: دفتر التشغيل النهائي

يُشغّل هذا الدفتر تطبيق **خُلاصة** من المستودع، ويجمع التلخيص التوليدي باستخدام AraT5v2 المدرّب واسترجاع تقارير التدقيق المنشورة عبر Google Fact Check Tools API. لا يتضمن تدريبًا أو تنزيلًا تلقائيًا للأوزان الخاصة.

يتطلب العرض الكامل وصولًا مصرحًا إلى مجلد النموذج وTokenizer على Google Drive. يُفضَّل Google Colab مع T4 GPU؛ يُحتفظ بتثبيت PyTorch/CUDA المتاح. يتطلب البحث المباشر مفتاح `FACT_CHECK_API_KEY` في Colab Secrets؛ لا يتطلب العرض المحفوظ مفتاحًا. GDELT وNLI معطّلان افتراضيًا.

**حالة التحقق:** وُثّق نجاح التلخيص والبحث المباشر في جلسة Colab خارجية أجرتها المطوّرة. فُحص هذا الدفتر من حيث البنية والصياغة واختبارات محاكاة؛ لا يُعد ذلك إعادة تنفيذ شاملة بالأوزان الخاصة. تبقى مخرجات الدفتر المحفوظة فارغة لحماية الأسرار والمسارات.

## 1. تهيئة بيئة التنفيذ ومصدر التطبيق

يُستخدم الفرع `feature/fact-checking` الذي يتضمن الوظائف الحالية، دون تعديل `main`. تُعرض بصمة الإصدار الفعلية لتوثيق التشغيل. يُمنع تحديث نسخة تحتوي تعديلات محلية، ويُطلب إعادة تشغيل الجلسة إذا سبق تحميل وحدات التطبيق. لا تُحذف مجلدات ولا يُستخدم `reset --hard`.

In [ ]:
import os
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["KHULASA_FACTCHECK_ENABLED"] = "1"
os.environ["KHULASA_GDELT_ENABLED"] = "0"
os.environ["KHULASA_NLI_ENABLED"] = "0"

REPO_URL = "https://github.com/nada-ali-13/Arabic-news-summarizer.git"
CODE_REF = "feature/fact-checking"
REPO_DIR = Path("/content/khulasa-final")
FULL_DEMO = True  # يتطلب العرض الكامل ملفات النموذج المصرح بها.
RUN_SUMMARY_SMOKE = False  # اختبار اختياري على نص اصطناعي معلن.
SHARE_PUBLIC = True  # رابط Gradio عام مؤقت، وليس استضافة دائمة.


def prepare_checkout(repo_dir, repo_url, ref):
    repo_dir = Path(repo_dir)
    if repo_dir.exists():
        if not (repo_dir / ".git").is_dir():
            raise RuntimeError("مجلد التشغيل موجود وليس مستودع Git؛ يلزم اختيار مجلد آخر.")
        def git_read(*args):
            return subprocess.check_output(
                ["git", "-C", str(repo_dir), *args], text=True
            ).strip()
        if git_read("remote", "get-url", "origin") != repo_url:
            raise RuntimeError("مصدر المجلد الموجود مختلف؛ لم يُعدّل المجلد.")
        if git_read("status", "--porcelain"):
            raise RuntimeError("توجد تغييرات محلية؛ يلزم حفظها قبل تحديث المصدر.")
        if git_read("branch", "--show-current") != ref:
            raise RuntimeError("الفرع الحالي مختلف؛ لم يُبدّل الفرع تلقائيًا.")
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", ref], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "merge", "--ff-only", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", ref, "--single-branch",
                        repo_url, str(repo_dir)], check=True)
    return subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
    ).strip()


if "app" in sys.modules or any(n.startswith("arabic_summarizer") for n in sys.modules):
    raise RuntimeError("يلزم إعادة تشغيل الجلسة قبل تحديث المصدر وإعادة تشغيل الدفتر.")
SOURCE_COMMIT = prepare_checkout(REPO_DIR, REPO_URL, CODE_REF)
print("إصدار مصدر التطبيق:", SOURCE_COMMIT)

# تثبيت الاعتمادات مع تثبيت إصدار PyTorch الموجود دون استبداله.
torch_before = version("torch")
with tempfile.TemporaryDirectory(prefix="khulasa-install-") as install_dir:
    constraint = Path(install_dir) / "constraints.txt"
    constraint.write_text("torch==" + torch_before + "\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR),
                    "-c", str(constraint)], check=True)
assert version("torch") == torch_before, "تغيّر إصدار PyTorch؛ يلزم مراجعة بيئة التنفيذ."
assert version("gradio") == "5.49.1", "إصدار Gradio غير متوافق مع التطبيق."
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
print("اكتمل إعداد المكتبات؛ إصدار Gradio:", version("gradio"))

## 2. ربط التخزين وإعداد السر

يُربط Google Drive دون تعديل ملفاته. يُقرأ المفتاح من Colab Secrets فقط، ولا تُطبع قيمته أو عناوين طلبات تتضمنه. عند غيابه يستمر التلخيص والعرض المحفوظ، بينما يتعذر البحث المباشر. يجب عدم حفظ مخرجات تتضمن بيانات خاصة عند مشاركة نسخة من الدفتر.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    secret = userdata.get("FACT_CHECK_API_KEY")
except Exception:
    secret = None
if secret:
    os.environ["FACT_CHECK_API_KEY"] = secret
    print("مفتاح خدمة التدقيق متاح؛ لا تُعرض قيمته.")
else:
    os.environ.pop("FACT_CHECK_API_KEY", None)
    print("مفتاح خدمة التدقيق غير متاح؛ البحث المباشر غير مهيّأ، والعرض المحفوظ متاح.")
del secret

## 3. التحقق من ملفات النموذج وتحميله

يُدخل مسار المجلد المصرح به أثناء التنفيذ دون تضمينه في المصدر أو طباعته. يتطلب المحمّل الحالي ملفات **safetensors**، و`config.json`، و`tokenizer_config.json`، و`spiece.model`. في التصدير المجزأ، يُتحقق من جميع الأجزاء المشار إليها في ملف الفهرس. لا يُعد وجود هذه الملفات إثباتًا لجودة النموذج أو مصدر تدريبه؛ يجب أن تكون نسخة النموذج المدرّب المعتمدة.

يُحمّل النموذج قبل إطلاق العرض الكامل، ولا توجد بدائل صامتة إلى النموذج الأساسي. `FULL_DEMO=False` يتيح فحص الواجهة والتدقيق دون الأوزان؛ لا يتيح تلخيصًا إنتاجيًا. يُوصى بإعادة تشغيل الجلسة قبل تبديل الأوزان بعد تحميلها.

In [ ]:
import json
from getpass import getpass

from arabic_summarizer.inference import Summarizer


def validate_model_dir(model_dir):
    root = Path(model_dir).expanduser().resolve()
    required = ("config.json", "tokenizer_config.json", "spiece.model")
    if not root.is_dir() or not all(
        (root / name).is_file() and (root / name).stat().st_size > 0 for name in required
    ):
        raise ValueError("ملفات النموذج أو Tokenizer الأساسية غير مكتملة.")
    config = json.loads((root / "config.json").read_text(encoding="utf-8"))
    if config.get("model_type") != "t5":
        raise ValueError("إعدادات النموذج لا تشير إلى بنية T5 المتوقعة.")
    index_path = root / "model.safetensors.index.json"
    if index_path.is_file():
        index = json.loads(index_path.read_text(encoding="utf-8"))
        shards = set(index.get("weight_map", {}).values())
        if not shards:
            raise ValueError("فهرس أجزاء الأوزان فارغ.")
        for name in shards:
            shard = (root / name).resolve()
            if (shard.parent != root or not name.endswith(".safetensors")
                    or not shard.is_file() or shard.stat().st_size == 0):
                raise ValueError("أجزاء الأوزان غير مكتملة أو يتضمن الفهرس مسارًا غير مسموح.")
    elif not (root / "model.safetensors").is_file() or (root / "model.safetensors").stat().st_size == 0:
        raise ValueError("ملف model.safetensors أو فهرس الأجزاء مفقود.")
    return root


summarizer = None
if FULL_DEMO:
    private_model_path = getpass("مسار مجلد النموذج المصرح به (إدخال مخفي): ").strip()
    if not private_model_path:
        raise ValueError("يلزم تحديد مجلد الأوزان للعرض الكامل.")
    try:
        MODEL_DIR = validate_model_dir(private_model_path)
        os.environ["ARABIC_SUMMARIZER_MODEL"] = str(MODEL_DIR)
        summarizer = Summarizer(model_path=str(MODEL_DIR))
        summarizer.load()
    except Exception:
        raise RuntimeError("تعذر تحميل النموذج المحلي؛ يلزم فحص اكتمال الملفات والتوافق والذاكرة.") from None
    finally:
        del private_model_path
    print("اكتمل تحميل النموذج المدرّب؛ جهاز التنفيذ:", summarizer.device)
else:
    print("تشغيل الواجهة والتدقيق دون تحميل أوزان التلخيص؛ التلخيص الكامل غير متاح.")

## 4. اختبار توليد اختياري

النص التالي **اصطناعي لأغراض اختبار التشغيل** وليس خبرًا موثقًا. لا تُحسب منه مقاييس جودة ولا صحة وقائع. يتطلب تفعيل `RUN_SUMMARY_SMOKE` وجود نموذج محمّل، ولا يستبدل بروتوكول ROUGE الموثق في المستودع.

In [ ]:
SMOKE_ARTICLE = (
    "يتناول هذا النص الاصطناعي برنامجًا تدريبيًا افتراضيًا في البرمجة وتحليل البيانات. "
    "يشمل البرنامج محاضرات نظرية وتمارين تطبيقية ومشروعات قصيرة، مع جلسات للنقاش "
    "ومراجعة التقدم. وقد أُعد النص لاختبار عمل التوليد فقط، ولا يصف حدثًا موثقًا."
)
if RUN_SUMMARY_SMOKE:
    if summarizer is None:
        raise RuntimeError("يتطلب اختبار التلخيص تفعيل العرض الكامل وتحميل النموذج.")
    smoke_result = summarizer.summarize(SMOKE_ARTICLE, preference="قصير")
    print("ناتج اختبار التوليد على النص الاصطناعي:", smoke_result["summary"])
else:
    print("اختبار التوليد الاختياري غير مفعّل.")

## 5. تشغيل التطبيق وحالات العرض

يستخدم التطبيق الدالة `build_app()` من المصدر المثبّت، مع تمرير النموذج المحمّل صراحةً. يُغلق مثيل الواجهة السابق في الجلسة قبل إعادة الإطلاق لتجنب تراكم الخوادم. الرابط العام مؤقت ومرتبط باستمرار جلسة Colab؛ لا تُدخل فيه بيانات حساسة.

### مثال تقرير منشور

الادعاء: **لقاحات كوفيد-19 تسببت بتفشي فيروس هانتا**.

يرتبط هذا المثال بتقرير [في ميزان فرانس برس](https://factcheckarabic.afp.com/doc.afp.com.B3FY2T3)، وتقييمه الأصلي **مضلّل**. وثّق اختبار Colab الخارجي استرجاعه في 2026-11-09. لا يضمن ذلك استمرار وجود النتيجة نفسها في كل طلب لاحق.

للبحث المباشر يُترك وضع العرض المحفوظ غير محدد. عند تفعيله تُعرض نسخة عامة محفوظة من التقرير، دون بحث مباشر أو تحقق من إدخال جديد. في المقال متعدد الادعاءات يظهر ملخص واحد وأعداد الادعاءات والتقارير المطابقة، وتُطوى التفاصيل افتراضيًا. غياب تقرير لا يثبت صحة الادعاء أو خطأه؛ ولا تُستخلص نتيجة كلية للمقال من جزء من ادعاءاته.

In [ ]:
from app import build_app

if "demo_app" in globals():
    globals()["demo_app"].close()
demo_app = build_app(summarizer=summarizer)
demo_app.launch(share=SHARE_PUBLIC, inline=True, show_error=False)

## حدود التشغيل وإعادة الإنتاج

- أوزان AraT5v2 المضبوطة غير عامة، ولا تُحمّل أو تُدرّب بواسطة هذا الدفتر.
- يعتمد البحث المباشر على المفتاح والشبكة وحصة Google وتوافر تقارير مناسبة.
- AraT5v2 للتلخيص، وليس لتصنيف حقيقة الادعاءات. Google يسترجع تقييمات الجهات المنشورة ولا يولّد حكمًا مستقلًا.
- لا يُفعّل GDELT أو NLI في هذا المسار. اختبار NLI العربي المحدود الموثق منفصل عن تقويم التحقق الإنتاجي.
- التثبيت يقيد PyTorch بإصداره الموجود، ويثبت Gradio 5.49.1؛ قد تتغير الاعتمادات العابرة المتوافقة. تُسجل بصمة المصدر عند التشغيل.
- يجب محو المخرجات قبل مشاركة الدفتر؛ الأسرار لا توضع في المصدر، ومسار النموذج يُدخل بطريقة مخفية.

**تصميم وتطوير: ندى علي.**